<a href="https://colab.research.google.com/github/AiWriter404/100-Days-Python/blob/main/clinic_bot_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clinic Bot

Converted from `.py` to Jupyter Notebook format.

In [ ]:
"""
🏥 CLINIC & DOCTOR WHATSAPP BOT
UAE Market | Arabic + English + Urdu
Appointment Booking + Doctor Info + Insurance FAQ + 7-Day Trial
"""

from flask import Flask, request
from twilio.twiml.messaging_response import MessagingResponse
from datetime import datetime, timedelta
import json, os

app = Flask(__name__)

# ══════════════════════════════════════════════
# ✅ CONFIG
# ══════════════════════════════════════════════
TWILIO_NUM   = "whatsapp:+14155238886"
YOUR_PAYMENT = "wise.com/pay/yourname"

TRIAL_DAYS  = 7
TRIAL_START = datetime.now()
TRIAL_END   = TRIAL_START + timedelta(days=TRIAL_DAYS)

# ── Clinic Info ──
CLINIC = {
    "name":      "Al Shifa Medical Center",
    "phone":     "+97142345678",
    "emergency": "+97150999888",
    "location":  "Al Barsha, Dubai (Near Mall of Emirates)",
    "maps":      "maps.google.com/?q=AlShifaDubai",
    "email":     "info@alshifa.ae",
    "timing": {
        "weekday": "Monday–Friday: 8am – 9pm",
        "weekend": "Saturday: 9am – 6pm | Sunday: Closed",
    },
    "insurance": "DAMAN, Neuron, AXA, Orient, Oman Insurance, Metlife, ADNIC"
}

DOCTORS = {
    "general": "Dr. Sara Ahmed (GP) — Sat–Thu 9am–5pm",
    "dental":  "Dr. Khalid Hassan (Dentist) — Mon/Wed/Fri 10am–7pm",
    "skin":    "Dr. Priya Sharma (Dermatologist) — Tue/Thu 11am–6pm",
    "child":   "Dr. Omar Malik (Pediatrician) — Sat–Thu 8am–4pm",
    "female":  "Dr. Fatima Al Zaabi (Gynecologist) — Mon/Wed/Sat 9am–5pm",
}

APPOINTMENTS_FILE = "appointments.json"

def save_appointment(phone, name="", doctor="", date=""):
    appts = []
    if os.path.exists(APPOINTMENTS_FILE):
        with open(APPOINTMENTS_FILE) as f:
            appts = json.load(f)
    appts.append({
        "phone": phone, "name": name,
        "doctor": doctor, "date": date,
        "time": datetime.now().strftime("%Y-%m-%d %H:%M")
    })
    with open(APPOINTMENTS_FILE, "w") as f:
        json.dump(appts, f, indent=2)

# ══════════════════════════════════════════════
# 💬 REPLIES — 3 Languages
# ══════════════════════════════════════════════
REPLIES = {
    "greeting": {
        "en": (
            f"👋 *Welcome to {CLINIC['name']}!*\n\n"
            "How can I help you today?\n\n"
            "Reply with:\n"
            "• *book* — Book appointment\n"
            "• *doctors* — Our doctors\n"
            "• *timing* — Clinic hours\n"
            "• *insurance* — Insurance we accept\n"
            "• *fees* — Consultation fees\n"
            "• *location* — Find us\n"
            "• *emergency* — Emergency contact\n"
            "• *report* — Lab/test results\n"
            "• *human* — Speak to receptionist"
        ),
        "ar": (
            f"👋 *مرحباً بكم في {CLINIC['name']}!*\n\n"
            "كيف يمكنني مساعدتك؟\n\n"
            "رد بـ:\n"
            "• *حجز* — حجز موعد\n"
            "• *أطباء* — أطباؤنا\n"
            "• *مواعيد* — ساعات العمل\n"
            "• *تأمين* — التأمين المقبول\n"
            "• *رسوم* — رسوم الاستشارة\n"
            "• *موقع* — اعثر علينا\n"
            "• *طوارئ* — اتصال الطوارئ\n"
            "• *بشري* — تحدث مع الاستقبال"
        ),
        "ur": (
            f"👋 *{CLINIC['name']} میں خوش آمدید!*\n\n"
            "میں کیسے مدد کر سکتا ہوں؟\n\n"
            "جواب دیں:\n"
            "• *بک* — اپوائنٹمنٹ بک کریں\n"
            "• *ڈاکٹر* — ہمارے ڈاکٹر\n"
            "• *وقت* — کلینک کے اوقات\n"
            "• *انشورنس* — قابل قبول انشورنس\n"
            "• *فیس* — مشاورتی فیس\n"
            "• *جگہ* — ہمیں تلاش کریں\n"
            "• *ایمرجنسی* — ایمرجنسی رابطہ\n"
            "• *انسان* — ریسپشن سے بات کریں"
        )
    },

    "book": {
        "en": (
            f"📅 *Book an Appointment*\n\n"
            "Please reply with:\n"
            "1️⃣ Your full name\n"
            "2️⃣ Type of doctor needed:\n"
            "   • General (GP)\n"
            "   • Dental\n"
            "   • Skin\n"
            "   • Child/Pediatric\n"
            "   • Female/Gynecology\n"
            "3️⃣ Preferred date & time\n\n"
            f"📞 Or call: {CLINIC['phone']}\n"
            "⏰ Our team confirms within 30 minutes!"
        ),
        "ar": (
            "📅 *حجز موعد*\n\n"
            "يرجى الرد بـ:\n"
            "1️⃣ اسمك الكامل\n"
            "2️⃣ نوع الطبيب:\n"
            "   • طب عام\n"
            "   • أسنان\n"
            "   • جلدية\n"
            "   • أطفال\n"
            "   • نساء\n"
            "3️⃣ التاريخ والوقت المفضل\n\n"
            f"📞 أو اتصل: {CLINIC['phone']}\n"
            "⏰ فريقنا يؤكد خلال 30 دقيقة!"
        ),
        "ur": (
            "📅 *اپوائنٹمنٹ بک کریں*\n\n"
            "براہ کرم جواب دیں:\n"
            "1️⃣ آپ کا پورا نام\n"
            "2️⃣ ڈاکٹر کی قسم:\n"
            "   • جنرل (GP)\n"
            "   • دانتوں کا\n"
            "   • جلد\n"
            "   • بچوں کا\n"
            "   • خواتین\n"
            "3️⃣ پسندیدہ تاریخ و وقت\n\n"
            f"📞 یا کال کریں: {CLINIC['phone']}\n"
            "⏰ ہماری ٹیم 30 منٹ میں تصدیق کرتی ہے!"
        )
    },

    "doctors": {
        "en": (
            "👨‍⚕️ *Our Doctors:*\n\n"
            f"🩺 {DOCTORS['general']}\n"
            f"🦷 {DOCTORS['dental']}\n"
            f"✨ {DOCTORS['skin']}\n"
            f"👶 {DOCTORS['child']}\n"
            f"👩‍⚕️ {DOCTORS['female']}\n\n"
            "Reply *book* to book with any doctor!"
        ),
        "ar": (
            "👨‍⚕️ *أطباؤنا:*\n\n"
            f"🩺 د. سارة أحمد (طب عام)\n"
            f"🦷 د. خالد حسن (أسنان)\n"
            f"✨ د. بريا شارما (جلدية)\n"
            f"👶 د. عمر مالك (أطفال)\n"
            f"👩‍⚕️ د. فاطمة الزعابي (نساء)\n\n"
            "رد بـ *حجز* لحجز موعد!"
        ),
        "ur": (
            "👨‍⚕️ *ہمارے ڈاکٹر:*\n\n"
            f"🩺 ڈاکٹر سارہ احمد (جنرل)\n"
            f"🦷 ڈاکٹر خالد حسن (دانت)\n"
            f"✨ ڈاکٹر پریا شرما (جلد)\n"
            f"👶 ڈاکٹر عمر ملک (بچے)\n"
            f"👩‍⚕️ ڈاکٹر فاطمہ الزعابی (خواتین)\n\n"
            "*بک* لکھیں اپوائنٹمنٹ کے لیے!"
        )
    },

    "timing": {
        "en": (
            f"⏰ *Clinic Hours:*\n\n"
            f"📅 {CLINIC['timing']['weekday']}\n"
            f"📅 {CLINIC['timing']['weekend']}\n\n"
            f"📍 {CLINIC['location']}\n"
            f"📞 {CLINIC['phone']}"
        ),
        "ar": (
            "⏰ *ساعات العمل:*\n\n"
            "📅 الاثنين–الجمعة: 8ص – 9م\n"
            "📅 السبت: 9ص – 6م | الأحد: مغلق\n\n"
            f"📍 {CLINIC['location']}\n"
            f"📞 {CLINIC['phone']}"
        ),
        "ur": (
            "⏰ *کلینک کے اوقات:*\n\n"
            "📅 پیر–جمعہ: صبح 8 – رات 9\n"
            "📅 ہفتہ: صبح 9 – شام 6 | اتوار: بند\n\n"
            f"📍 {CLINIC['location']}\n"
            f"📞 {CLINIC['phone']}"
        )
    },

    "insurance": {
        "en": (
            "🏥 *Insurance We Accept:*\n\n"
            f"✅ {CLINIC['insurance']}\n\n"
            "Please bring your insurance card on your visit.\n"
            "For pre-approval queries: 📞 " + CLINIC['phone']
        ),
        "ar": (
            "🏥 *التأمين المقبول:*\n\n"
            f"✅ {CLINIC['insurance']}\n\n"
            "يرجى إحضار بطاقة التأمين عند الزيارة.\n"
            "للاستفسار: 📞 " + CLINIC['phone']
        ),
        "ur": (
            "🏥 *قابل قبول انشورنس:*\n\n"
            f"✅ {CLINIC['insurance']}\n\n"
            "وزٹ پر انشورنس کارڈ ساتھ لائیں۔\n"
            "سوالات: 📞 " + CLINIC['phone']
        )
    },

    "fees": {
        "en": (
            "💰 *Consultation Fees:*\n\n"
            "🩺 General Practitioner — AED 150\n"
            "🦷 Dentist (consultation) — AED 100\n"
            "✨ Dermatologist — AED 200\n"
            "👶 Pediatrician — AED 180\n"
            "👩‍⚕️ Gynecologist — AED 200\n\n"
            "💳 Cash, Card & Insurance accepted\n"
            "📅 Reply *book* to book now!"
        ),
        "ar": (
            "💰 *رسوم الاستشارة:*\n\n"
            "🩺 طب عام — 150 درهم\n"
            "🦷 أسنان — 100 درهم\n"
            "✨ جلدية — 200 درهم\n"
            "👶 أطفال — 180 درهم\n"
            "👩‍⚕️ نساء — 200 درهم\n\n"
            "💳 نقدي، بطاقة وتأمين مقبول"
        ),
        "ur": (
            "💰 *مشاورتی فیس:*\n\n"
            "🩺 جنرل ڈاکٹر — AED 150\n"
            "🦷 دانتوں کا — AED 100\n"
            "✨ جلد — AED 200\n"
            "👶 بچوں کا — AED 180\n"
            "👩‍⚕️ خواتین — AED 200\n\n"
            "💳 کیش، کارڈ اور انشورنس قابل قبول"
        )
    },

    "location": {
        "en": (
            f"📍 *Find Us:*\n\n"
            f"{CLINIC['location']}\n\n"
            f"🗺️ Google Maps: {CLINIC['maps']}\n"
            f"📞 {CLINIC['phone']}\n\n"
            "Free parking available! 🅿️"
        ),
        "ar": (
            f"📍 *موقعنا:*\n\n"
            f"{CLINIC['location']}\n\n"
            f"🗺️ خرائط Google: {CLINIC['maps']}\n"
            f"📞 {CLINIC['phone']}\n\n"
            "موقف سيارات مجاني! 🅿️"
        ),
        "ur": (
            f"📍 *ہماری جگہ:*\n\n"
            f"{CLINIC['location']}\n\n"
            f"🗺️ گوگل میپس: {CLINIC['maps']}\n"
            f"📞 {CLINIC['phone']}\n\n"
            "مفت پارکنگ! 🅿️"
        )
    },

    "emergency": {
        "en":  f"🚨 *EMERGENCY*\n\nCall immediately:\n📞 {CLINIC['emergency']}\n\nOr go to nearest hospital A&E.",
        "ar":  f"🚨 *طوارئ*\n\nاتصل فوراً:\n📞 {CLINIC['emergency']}",
        "ur":  f"🚨 *ایمرجنسی*\n\nفوری کال کریں:\n📞 {CLINIC['emergency']}"
    },

    "report": {
        "en":  f"📋 *Lab Results & Reports*\n\nPlease call reception:\n📞 {CLINIC['phone']}\n📧 {CLINIC['email']}\n\nReady usually within 24–48 hours.",
        "ar":  f"📋 *نتائج المختبر*\n\nاتصل بالاستقبال:\n📞 {CLINIC['phone']}",
        "ur":  f"📋 *لیب رپورٹس*\n\nریسپشن کو کال کریں:\n📞 {CLINIC['phone']}"
    },

    "human": {
        "en":  f"👩‍💼 *Connecting to Reception...*\n\nCall us directly:\n📞 {CLINIC['phone']}\n⏰ Available during clinic hours",
        "ar":  f"👩‍💼 *جاري التحويل للاستقبال...*\n\nاتصل بنا مباشرة:\n📞 {CLINIC['phone']}",
        "ur":  f"👩‍💼 *ریسپشن سے جوڑ رہے ہیں...*\n\nہمیں کال کریں:\n📞 {CLINIC['phone']}"
    },

    "default": {
        "en":  f"Thank you! 🏥 Our team will assist shortly.\n📞 {CLINIC['phone']}\n\nReply *hi* for main menu.",
        "ar":  f"شكراً! 🏥 فريقنا سيساعدك قريباً.\n📞 {CLINIC['phone']}",
        "ur":  f"شکریہ! 🏥 ہماری ٹیم جلد مدد کرے گی۔\n📞 {CLINIC['phone']}"
    },

    "trial_warning": {
        "en": "⚠️ *Trial ends in {days} day(s)!*\nKeep bot active — Pay AED 499:\n👉 " + YOUR_PAYMENT,
        "ar": "⚠️ *تنتهي التجربة خلال {days} يوم!*\nادفع 499 درهم:\n👉 " + YOUR_PAYMENT,
        "ur": "⚠️ *ٹرائل {days} دن میں ختم!*\nAED 499 ادا کریں:\n👉 " + YOUR_PAYMENT
    },
    "trial_expired": {
        "en": "❌ Trial ended.\nReactivate bot: " + YOUR_PAYMENT,
        "ar": "❌ انتهت التجربة.\nللتفعيل: " + YOUR_PAYMENT,
        "ur": "❌ ٹرائل ختم۔\nدوبارہ چالو: " + YOUR_PAYMENT
    }
}

# ══════════════════════════════════════════════
# 🌍 LANGUAGE DETECTION
# ══════════════════════════════════════════════
def detect_lang(text):
    arabic = sum(1 for c in text if '\u0600' <= c <= '\u06FF')
    urdu_words = ["helo","hello","kya","hai","book","doctor","timing","fees",
                  "waqt","fee","beemar","marz","dard","insurance","location","kahan"]
    if arabic > 2: return "ar"
    if any(w in text.lower() for w in urdu_words): return "ur"
    return "en"

# ══════════════════════════════════════════════
# 🤖 SMART REPLY
# ══════════════════════════════════════════════
def get_reply(msg, lang, user_phone):
    m = msg.lower()

    greet     = ["hi","hello","helo","hey","salam","مرحبا","السلام","ہیلو","start","menu"]
    book_w    = ["book","appointment","appt","حجز","موعد","بک","اپوائنٹ"]
    doctor_w  = ["doctor","doc","specialist","أطباء","طبيب","ڈاکٹر","physician"]
    timing_w  = ["timing","time","hours","open","close","مواعيد","وقت","اوقات","کب"]
    insure_w  = ["insurance","cover","taamin","تأمين","انشورنس","card"]
    fees_w    = ["fee","fees","cost","price","charge","رسوم","فیس","kitna"]
    location_w= ["location","address","where","map","موقع","جگہ","kahan","directions"]
    emerg_w   = ["emergency","urgent","emerg","طوارئ","ایمرجنسی","accident"]
    report_w  = ["report","result","lab","test","نتيجة","رپورٹ","نتائج"]
    human_w   = ["human","person","reception","staff","بشري","انسان","receptionist"]

    if any(w in m for w in book_w):
        save_appointment(user_phone)
        return REPLIES["book"][lang]

    if any(w in m for w in greet):     return REPLIES["greeting"][lang]
    if any(w in m for w in doctor_w):  return REPLIES["doctors"][lang]
    if any(w in m for w in timing_w):  return REPLIES["timing"][lang]
    if any(w in m for w in insure_w):  return REPLIES["insurance"][lang]
    if any(w in m for w in fees_w):    return REPLIES["fees"][lang]
    if any(w in m for w in location_w):return REPLIES["location"][lang]
    if any(w in m for w in emerg_w):   return REPLIES["emergency"][lang]
    if any(w in m for w in report_w):  return REPLIES["report"][lang]
    if any(w in m for w in human_w):   return REPLIES["human"][lang]

    return REPLIES["default"][lang]

# ══════════════════════════════════════════════
# ⏳ TRIAL
# ══════════════════════════════════════════════
def is_active():  return datetime.now() < TRIAL_END
def days_left():  return max(0, (TRIAL_END - datetime.now()).days)

# ══════════════════════════════════════════════
# 🤖 MAIN ENDPOINT
# ══════════════════════════════════════════════
@app.route("/clinic", methods=["POST"])
def clinic_bot():
    user_msg   = request.form.get("Body", "").strip()
    user_phone = request.form.get("From", "")
    resp       = MessagingResponse()
    lang       = detect_lang(user_msg)

    if not is_active():
        resp.message(REPLIES["trial_expired"][lang])
        return str(resp)

    base = get_reply(user_msg, lang, user_phone)
    d    = days_left()
    if d <= 2:
        base += "\n\n" + REPLIES["trial_warning"][lang].format(days=d)

    resp.message(base)
    return str(resp)

@app.route("/", methods=["GET"])
def home():
    return f"🏥 Clinic Bot | {'✅ ACTIVE' if is_active() else '❌ EXPIRED'} | {days_left()} days left"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8080)
